# 02 — Automated outburst detection

Report section 4.2. Routes every source in the classified catalogue to one of four
detection configs (BH LXB / NS LXB / unknown LXB / HXB) and produces the main
outburst catalogue used by every later stage.

See `docs/algorithms.md` §2 for the detection logic.

In [ ]:
import sys
sys.path.insert(0, '../src')
import pandas as pd
import matplotlib.pyplot as plt

from xrb_pipeline.detection import tiered_detector as det

## Diagnostic: run the detector on one known source first

Always eyeball a couple of known sources before committing to a full-catalogue run.

In [ ]:
MAXI_ID = "J1911+005"   # Aql X-1

lc_df = det.fetch_maxi_lightcurve(MAXI_ID)
config = det.NS_LXB_CONFIG   # match this source's actual class
outbursts, baseline, threshold = det.detect_lxb_outbursts(lc_df, MAXI_ID, config)

print(f"{len(outbursts)} outburst(s) found. baseline={baseline:.4f} threshold={threshold:.4f}")

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(lc_df['mjd'], lc_df['flux_total'], color='steelblue', lw=0.7)
ax.axhline(baseline, color='gray', ls='--', lw=1, label='baseline')
ax.axhline(threshold, color='orange', ls='--', lw=1, label='threshold')
for ob in outbursts:
    ax.axvspan(ob['t_start'], ob['t_end'], color='red', alpha=0.15)
ax.set_xlabel('MJD'); ax.set_ylabel('Flux (ct/s/cm^2)')
ax.set_title(f'{MAXI_ID} -- detection diagnostic'); ax.legend()
plt.tight_layout(); plt.show()

## Full-catalogue run

Once the diagnostic above looks right, run the whole classified catalogue.

In [ ]:
CLASSIFIED_CSV = "step1b_maxi_xrb_classified.csv"
OUTDIR = "../data/detection"

# ob_df = det.run_pipeline(CLASSIFIED_CSV, OUTDIR)
print("Uncomment above once step1b_maxi_xrb_classified.csv is ready (from notebook 01).")

Outputs: `step2_outbursts.csv` (confirmed outbursts), `step2_tier0_failed.csv`,
`step2_skipped_persistent.csv`. Feeds notebooks 03 (fitting) and 04 (duration stats).